# Decode text from stored Whisper encoder embeddings

The feature HDF5 files (`data/features/{split}/<subject>/whisper_features_{split}.hdf5`) store, per trial:

- `encoder_embedding` — Whisper `tiny.en` encoder output, shape `(1500, 384)`
- `audio_length` — number of 16 kHz audio samples
- `transcription` — ground-truth sentence

Whisper's `decode()` detects when its input already has shape `(n_audio_ctx, n_audio_state)` = `(1500, 384)` and **skips the encoder**, using the array directly as audio features. So we can feed the stored embeddings straight into the decoder to reconstruct text.

**Length masking.** The encoder pads every clip to Whisper's fixed 30 s window, so the 1500 frames usually contain many trailing frames that only encode silence. Using `audio_length` we keep just the frames that cover real audio and zero-pad the rest. At 16 kHz with hop length 160 and the encoder's 2x downsampling, each encoder frame spans `160 * 2 = 320` audio samples (20 ms), so the number of valid frames is `ceil(audio_length / 320)`.

In [7]:
import glob
import math
import h5py
import numpy as np
import torch
import whisper

MODEL_NAME = "tiny.en"  # must match the model used to extract the features
FEATURES_DIR = "data/features"
FRAME_SAMPLES = whisper.audio.HOP_LENGTH * 2  # 320: audio samples per encoder frame

device = "cuda" if torch.cuda.is_available() else "cpu"
use_fp16 = device == "cuda"
model = whisper.load_model(MODEL_NAME, device=device)
N_CTX = model.dims.n_audio_ctx
print(f"Loaded {MODEL_NAME} on {device} | n_audio_ctx={N_CTX}, n_audio_state={model.dims.n_audio_state}")
print(f"audio samples per encoder frame: {FRAME_SAMPLES}")

Loaded tiny.en on cpu | n_audio_ctx=1500, n_audio_state=384
audio samples per encoder frame: 320


In [8]:
def valid_frames(audio_length):
    """Number of encoder frames that cover real audio (rest is padding)."""
    return max(1, min(N_CTX, math.ceil(int(audio_length) / FRAME_SAMPLES)))


def mask_embedding(embedding, audio_length):
    """Keep only the valid context frames; zero-pad the rest. Shape stays (1500, 384)."""
    n_valid = valid_frames(audio_length)
    masked = np.zeros_like(embedding)
    masked[:n_valid] = embedding[:n_valid]
    return masked, n_valid


def decode_embedding(embedding):
    """Decode a single (1500, 384) encoder embedding to text via the Whisper decoder."""
    feats = torch.as_tensor(embedding, dtype=torch.float32, device=device).unsqueeze(0)
    expected = (model.dims.n_audio_ctx, model.dims.n_audio_state)
    if tuple(feats.shape[-2:]) != expected:
        raise ValueError(f"embedding shape {tuple(feats.shape[-2:])} != expected {expected}")
    options = whisper.DecodingOptions(language="en", without_timestamps=True, fp16=use_fp16)
    result = whisper.decode(model, feats, options)
    result = result[0] if isinstance(result, list) else result
    return result.text.strip()


def read_trial(feature_file, trial_name):
    with h5py.File(feature_file, "r") as h:
        g = h[trial_name]
        emb = g["encoder_embedding"][()]
        audio_length = int(g["audio_length"][()])
        truth = g["transcription"][()]
        truth = truth.decode() if isinstance(truth, bytes) else str(truth)
    return emb, audio_length, truth

## Decode a few trials (length-masked embeddings)

In [9]:
SPLIT = "val"
feature_files = sorted(glob.glob(f"{FEATURES_DIR}/{SPLIT}/*/whisper_features_{SPLIT}.hdf5"))
print(f"{len(feature_files)} subject feature files for split '{SPLIT}'")

feature_file = feature_files[0]
with h5py.File(feature_file, "r") as h:
    trial_names = list(h.keys())
print(f"Using {feature_file} ({len(trial_names)} trials)\n")

for trial_name in trial_names[:8]:
    emb, audio_length, truth = read_trial(feature_file, trial_name)
    masked, n_valid = mask_embedding(emb, audio_length)
    pred = decode_embedding(masked)
    match = "OK " if pred.lower() == truth.lower() else "DIFF"
    print(f"[{match}] {trial_name} | {audio_length} samples -> {n_valid}/{N_CTX} valid frames")
    print(f"   truth: {truth}")
    print(f"   pred : {pred}\n")

41 subject feature files for split 'val'
Using data/features/val/t15.2023.08.13/whisper_features_val.hdf5 (35 trials)

[OK ] trial_0000 | 57567 samples -> 180/1500 valid frames
   truth: You can see the code at this point as well.
   pred : You can see the code at this point as well.

[OK ] trial_0001 | 40767 samples -> 128/1500 valid frames
   truth: How does it keep the cost down?
   pred : How does it keep the cost down?

[OK ] trial_0002 | 37567 samples -> 118/1500 valid frames
   truth: Not too controversial.
   pred : Not too controversial.

[OK ] trial_0003 | 55567 samples -> 174/1500 valid frames
   truth: The jury and a judge work together on it.
   pred : The jury and a judge work together on it.

[DIFF] trial_0004 | 41167 samples -> 129/1500 valid frames
   truth: Were quite vocal about it.
   pred : We're quite vocal about it.

[OK ] trial_0005 | 54767 samples -> 172/1500 valid frames
   truth: He said the decision to part ways was mutual.
   pred : He said the decision to 

[OK ] trial_0000 | 57567 samples -> 180/1500 valid frames
   truth: You can see the code at this point as well.
   pred : You can see the code at this point as well.

[OK ] trial_0001 | 40767 samples -> 128/1500 valid frames
   truth: How does it keep the cost down?
   pred : How does it keep the cost down?

[OK ] trial_0002 | 37567 samples -> 118/1500 valid frames
   truth: Not too controversial.
   pred : Not too controversial.



[OK ] trial_0003 | 55567 samples -> 174/1500 valid frames
   truth: The jury and a judge work together on it.
   pred : The jury and a judge work together on it.

[DIFF] trial_0004 | 41167 samples -> 129/1500 valid frames
   truth: Were quite vocal about it.
   pred : We're quite vocal about it.

[OK ] trial_0005 | 54767 samples -> 172/1500 valid frames
   truth: He said the decision to part ways was mutual.
   pred : He said the decision to part ways was mutual.



[DIFF] trial_0006 | 51967 samples -> 163/1500 valid frames
   truth: In fact this morning when they were talking.
   pred : In fact, this morning when they were talking.

[OK ] trial_0007 | 42367 samples -> 133/1500 valid frames
   truth: This is like a cruelty joke.
   pred : This is like a cruelty joke.



## Full vs length-masked: WER over a sample

Compares decoding the raw 1500-frame embedding against the length-masked version.

In [4]:
def wer(reference, hypothesis):
    """Word error rate via Levenshtein distance over word tokens."""
    r = reference.lower().split()
    hyp = hypothesis.lower().split()
    d = [[0] * (len(hyp) + 1) for _ in range(len(r) + 1)]
    for i in range(len(r) + 1):
        d[i][0] = i
    for j in range(len(hyp) + 1):
        d[0][j] = j
    for i in range(1, len(r) + 1):
        for j in range(1, len(hyp) + 1):
            cost = 0 if r[i - 1] == hyp[j - 1] else 1
            d[i][j] = min(d[i - 1][j] + 1, d[i][j - 1] + 1, d[i - 1][j - 1] + cost)
    return d[len(r)][len(hyp)] / max(len(r), 1)


N = 30
stats = {"full": {"exact": 0, "wers": []}, "masked": {"exact": 0, "wers": []}}
for trial_name in trial_names[:N]:
    emb, audio_length, truth = read_trial(feature_file, trial_name)
    masked, _ = mask_embedding(emb, audio_length)
    for key, arr in (("full", emb), ("masked", masked)):
        pred = decode_embedding(arr)
        stats[key]["exact"] += int(pred.lower() == truth.lower())
        stats[key]["wers"].append(wer(truth, pred))

n = N
print(f"trials evaluated: {n}\n")
for key in ("full", "masked"):
    ex = stats[key]["exact"]
    mw = sum(stats[key]["wers"]) / n
    print(f"{key:7s} | exact {ex}/{n} ({100 * ex / n:.1f}%) | mean WER {mw:.4f}")

trials evaluated: 30

full    | exact 24/30 (80.0%) | mean WER 0.0539
masked  | exact 26/30 (86.7%) | mean WER 0.0206


## Decode an arbitrary trial (length-masked)

Set `subject`, `split`, and `trial` to decode any single stored embedding.

In [5]:
split = "val"
subject = None   # e.g. "t15.2023.08.13"; None picks the first subject
trial = "trial_0000"

if subject is None:
    feature_file = sorted(glob.glob(f"{FEATURES_DIR}/{split}/*/whisper_features_{split}.hdf5"))[0]
else:
    feature_file = f"{FEATURES_DIR}/{split}/{subject}/whisper_features_{split}.hdf5"

emb, audio_length, truth = read_trial(feature_file, trial)
masked, n_valid = mask_embedding(emb, audio_length)
pred = decode_embedding(masked)
print(f"file : {feature_file}")
print(f"trial: {trial} | {audio_length} samples -> {n_valid}/{N_CTX} valid frames")
print(f"truth: {truth}")
print(f"pred : {pred}")

file : data/features/val/t15.2023.08.13/whisper_features_val.hdf5
trial: trial_0000 | 57567 samples -> 180/1500 valid frames
truth: You can see the code at this point as well.
pred : You can see the code at this point as well.
